# Improved Neural Network to Diagnose Diabetes

In [12]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from scipy import stats
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from opacus import PrivacyEngine
from opacus.utils.batch_memory_manager import BatchMemoryManager

df = pd.read_csv("../data/diabetes_dataset.csv")

In [13]:
df_clean = df.drop(columns=[
    'diabetes_stage', 'diabetes_risk_score'])

In [14]:
# split features (X) and target (y)
X = df_clean.drop('diagnosed_diabetes', axis=1)
X = pd.get_dummies(X, drop_first=True)
y = df_clean['diagnosed_diabetes']

# test cases
print(f"Shape of X: {X.shape}")
print(f"Shape of y: {y.shape}")

# create train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# standardize numerical features using StandardScaler
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

Shape of X: (100000, 40)
Shape of y: (100000,)


In [20]:
# convert the numpy arrays to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

# create DataLoaders for batch processing
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

In [21]:
class DiabetesNN(nn.Module):
    def __init__(self, input_dim):
        super(DiabetesNN, self).__init__()
        # first layer is input to hidden
        self.layer1 = nn.Linear(input_dim, 256)
        # second is hidden to hidden
        self.layer2 = nn.Linear(256, 128)
        # third is hidden to hidden
        self.layer3 = nn.Linear(128, 64)
        # fourth is hidden to output
        self.layer4 = nn.Linear(64, 1)
        self.dropout = nn.Dropout(0.2)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.dropout(x)
        
        x = self.layer2(x)
        x = self.relu(x)
        x = self.dropout(x)
        
        x = self.layer3(x)
        x = self.relu(x)

        x = self.layer4(x)
        x = self.sigmoid(x)
        return x

In [22]:
# initialize model
input_dim = X.shape[1]
model_sgd = DiabetesNN(X.shape[1])

# define optimizer and loss
criterion = nn.BCELoss()

optimizer_sgd = optim.SGD(model_sgd.parameters(), lr=0.01, momentum=0.9)

num_epochs = 50

print("Training for Non-DP SGD")
for epoch in range(num_epochs):
    model_sgd.train()
    running_loss = 0.0
    
    for inputs, labels in train_loader:
        optimizer_sgd.zero_grad()
        outputs = model_sgd(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_sgd.step()
        
        running_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader):.4f}")

# Print out the accuracy:
model_sgd.eval()
with torch.no_grad():
    outputs = model_sgd(X_test_tensor)
    predicted = (outputs > 0.5).float()
    accuracy = (predicted == y_test_tensor).sum() / y_test_tensor.size(0)
    print(f"\nStandard SGD Test Accuracy: {accuracy.item() * 100:.2f}%")

Training for Non-DP SGD
Epoch 10, Loss: 0.2413
Epoch 20, Loss: 0.2273
Epoch 30, Loss: 0.2218
Epoch 40, Loss: 0.2168
Epoch 50, Loss: 0.2138

Standard SGD Test Accuracy: 91.25%
